# 🏛 ConfAIr - Conference Assistant

A multi-agent AI system that:
1. Understands a research paper (PDF or text)
2. Finds suitable conferences with upcoming deadlines
3. Checks conference reputation / predatory risk
4. Generates a detailed, venue-aware peer review

Track: **Enterprise Agents**
Built with: **Gemini 2.5 + Google ADK + custom tools**

## 0. Install dependencies

In [3]:
!pip install -q pypdf

## 1. API setup & imports

In [4]:
import os
from datetime import datetime, date
from pathlib import Path
from typing import Optional, Dict, Any, List

from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    # Use public GenAI, not Vertex AI
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
    print("✅ Google GenAI API key configured.")
except Exception as e:
    print(
        "🔑 Authentication Error: Please add 'GOOGLE_API_KEY' to your Kaggle secrets.\n"
        f"Details: {e}"
    )

✅ Google GenAI API key configured.


In [5]:
from google.genai import types

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool

import textwrap

print("✅ ADK components imported.")

✅ ADK components imported.


## 2. Model & retry config

In [6]:
retry_config = types.HttpRetryOptions(
    attempts=5,
    exp_base=7,
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],
)

gemini_model = Gemini(
    model="gemini-2.5-flash-lite",
    retry_options=retry_config,
)

print("✅ Gemini model initialized with retry config.")

✅ Gemini model initialized with retry config.


## 3. Tools
 
### 3.1 `pdf_to_text` – parse PDFs into text chunks

In [7]:
from pypdf import PdfReader

def pdf_to_text(
    path: str,
    max_pages: Optional[int] = 10,
) -> Dict[str, Any]:
    """
    Extracts text from a PDF file and returns a structured summary of the content.

    Args:
        path: Path to the PDF file (e.g., from /kaggle/input/...).
        max_pages: Maximum number of pages to parse (to avoid huge PDFs).
                   If None, parse all pages.

    Returns:
        Dict with:
          - status: "ok" or "error"
          - message: status explanation
          - metadata: {num_pages, parsed_pages, file_name}
          - full_text: concatenated text of parsed pages (may be truncated)
          - first_page_text: raw text of first page
          - title_guess: first non-empty line from the first page
          - abstract_guess: best-effort extraction of an abstract-like segment
    """
    try:
        pdf_path = Path(path)
        if not pdf_path.exists():
            return {
                "status": "error",
                "message": f"PDF path does not exist: {path}",
            }

        reader = PdfReader(str(pdf_path))
        num_pages = len(reader.pages)

        if max_pages is None:
            max_pages_to_read = num_pages
        else:
            max_pages_to_read = min(num_pages, max_pages)

        pages_text: List[str] = []
        for i in range(max_pages_to_read):
            page = reader.pages[i]
            text = page.extract_text() or ""
            pages_text.append(text)

        full_text = "\n\n".join(pages_text)
        first_page_text = pages_text[0] if pages_text else ""

        # Naive title guess: first non-empty line of first page
        title_guess = None
        for line in first_page_text.splitlines():
            clean = line.strip()
            if len(clean) > 5:
                title_guess = clean
                break

        # Naive abstract guess: look for the word "abstract"
        abstract_guess = None
        lower = full_text.lower()
        if "abstract" in lower:
            idx = lower.index("abstract")
            snippet = full_text[idx : idx + 2000]
            abstract_guess = snippet.split("\n\n")[0].strip()
        else:
            # fallback: first ~1500 chars of first page
            abstract_guess = first_page_text[:1500].strip()

        return {
            "status": "ok",
            "message": "PDF text extracted successfully.",
            "metadata": {
                "num_pages": num_pages,
                "parsed_pages": max_pages_to_read,
                "file_name": pdf_path.name,
            },
            "full_text": full_text,
            "first_page_text": first_page_text,
            "title_guess": title_guess,
            "abstract_guess": abstract_guess,
        }
    except Exception as e:
        return {
            "status": "error",
            "message": f"Failed to parse PDF: {e}",
        }

print("✅ pdf_to_text tool defined.")

✅ pdf_to_text tool defined.


### 3.2 `parse_deadline_and_days_left` – optional date sanity helper

In [8]:
def parse_deadline_and_days_left(deadline_text: str) -> Dict[str, Any]:
    """
    Helper for agents:
    - Try to parse a deadline date from arbitrary text (e.g., 'December 5, 2025')
    - Compute days_left relative to TODAY if parsing succeeds

    This is deliberately simple so the LLM can choose how to use it.
    """
    try:
        txt = deadline_text.strip()
        # Try ISO first
        try:
            dt = datetime.fromisoformat(txt).date()
        except Exception:
            dt = None
            for fmt in ("%B %d, %Y", "%b %d, %Y", "%d %B %Y", "%d %b %Y"):
                try:
                    dt = datetime.strptime(txt, fmt).date()
                    break
                except Exception:
                    continue

        if dt is None:
            return {
                "status": "error",
                "message": f"Could not parse deadline date from: {deadline_text}",
            }

        today = date.today()
        days_left = (dt - today).days

        return {
            "status": "ok",
            "deadline_iso": dt.isoformat(),
            "days_left": days_left,
            "is_future": days_left >= 0,
        }
    except Exception as e:
        return {
            "status": "error",
            "message": f"Error parsing deadline: {e}",
        }

print("✅ parse_deadline_and_days_left tool defined.")

✅ parse_deadline_and_days_left tool defined.


## 4. Specialist Agents

We now define four main specialist agents:
- PaperProfilerAgent
- ConferenceFinderAgent
- ConferenceReputationAgent
- PaperReviewAgent

### 4.1 PaperProfilerAgent – understand the paper

In [9]:
paper_profiler_agent = LlmAgent(
    name="paper_profiler_agent",
    model=gemini_model,
    instruction=textwrap.dedent(
        """
        You are a **Paper Profiler**.

        You will usually be called after the `pdf_to_text` tool has run.

        Input you will receive (in natural language):
        - title_guess
        - abstract_guess
        - optionally some truncated full_text
        - metadata (num_pages, etc.)

        YOUR JOB:
        1. Infer:
           - a clean, user-friendly title (or refine the guess)
           - a clean abstract (2–5 sentences)
           - the main research FIELD / AREA
             (e.g., "deep learning for medical image segmentation in oncology",
                   "bioinformatics for cancer genomics",
                   "NLP for low-resource languages").
           - the main METHOD(S) (e.g., U-Net, transformers, self-supervised learning).
           - the main CONTRIBUTIONS / CLAIMS (bullet points).

        2. Output a **structured JSON-like summary** in markdown, for example:

           {
             "title": "...",
             "abstract": "...",
             "field": "...",
             "methods": ["...","..."],
             "contributions": ["...","...","..."]
           }

        Be concise but informative.
        This summary will be used by other agents (conference finder, reviewer).

        If the input is short or incomplete, say that some fields are uncertain.
        """
    ),
    tools=[],  # pure LLM, no extra tools
)

paper_profiler_runner = InMemoryRunner(agent=paper_profiler_agent)

print("✅ Paper Profiler Agent & runner created.")

✅ Paper Profiler Agent & runner created.


### 4.2 ConferenceFinderAgent – find upcoming conferences (via web search)

In [10]:
conference_finder_agent = LlmAgent(
    name="conference_finder_websearch",
    model=gemini_model,
    instruction=textwrap.dedent(
        """
        You are a **Conference Finder Assistant** for researchers.

        Your job:
        - Given a description of the user's paper or research field,
          find *upcoming* conferences that are a good match.
        - You MUST use the `google_search` tool to get current information.
        - You MUST NOT invent conferences or deadlines.

        HOW TO WORK (SUMMARY):

        1. Interpret the FIELD / SUBJECT description.

        2. Call `google_search` with 1–3 queries like:
           - "<field> conference 2026 call for papers deadline"
           - "<field> top tier conference submission deadline"
           - "<field> CFP 2025 conference"

        3. For each relevant conference:
           - Extract:
             * name
             * acronym (if visible)
             * main field / topic
             * submission deadline (paper submission)
             * location
             * website URL
             * rough reputation: "top-tier", "strong", "mid-tier", "unknown"

        4. Deadline filtering:
           - Use today's date as reference.
           - If a deadline is clearly in the **past**, do NOT include it in
             the main "upcoming" table.
           - You may mention such venues in a separate "past but relevant"
             section, clearly marked as **deadline passed**.
           - If you don't know the deadline, write "unknown (check site)".

        5. Sorting:
           - If the user mentions **prestige / top-tier**, sort primarily by reputation,
             then by deadline (only among upcoming conferences).
           - If the user mentions **soon / earliest**, sort primarily by earliest
             upcoming deadline.
           - Otherwise, default to earliest upcoming deadline.

        6. Output:
           - Start with a short summary sentence.
           - Then show a Markdown table:

             | Rank | Acronym | Name | Field | Submission Deadline | Days Left | Reputation | Location | URL |

           - Only include upcoming conferences in that table.
           - For each:
             * use a best-effort "Days Left" estimate based on the date you see,
               but NEVER give positive days if the date is in the past.
             * if you are unsure, say "Unknown".

        7. Honesty:
           - Never hallucinate precise dates; mark them as "unknown (check site)" if needed.
           - Always end with:
             "⚠️ Always verify deadlines & details on the official conference website,
             as they can change."

        You may call `google_search` multiple times.
        """,
    ),
    tools=[google_search],
)

conference_finder_runner = InMemoryRunner(agent=conference_finder_agent)

print("✅ Conference Finder Agent & runner created.")

✅ Conference Finder Agent & runner created.


### 4.3 ConferenceReputationAgent – reputation & predatory risk

In [11]:
conference_reputation_agent = LlmAgent(
    name="conference_reputation_agent",
    model=gemini_model,
    instruction=textwrap.dedent(
        """
        You are a **Conference Reputation & Predatory Risk Analyzer**.

        Given a conference name (and optionally a URL), your job is to:
        - Assess how reputable the venue appears, using ONLY web evidence.
        - Identify possible predatory / low-quality signals.
        - Output a structured, honest assessment.

        You MUST use `google_search` with queries like:
        - "<conference name> official site"
        - "<conference name> publisher"
        - "<conference name> indexed by"
        - "<conference name> predatory"
        - "<conference name> scam"

        POSITIVE signals (examples):
        - Affiliation with respected bodies (IEEE, ACM, Springer, major societies).
        - Many past editions with clear history.
        - Mentions of indexing (Scopus, Web of Science, DBLP) in trusted sources.
        - Program committees with known researchers / universities.
        - Professional, detailed website with clear review process.

        NEGATIVE signals (examples):
        - Appears on known predatory lists or warning blogs.
        - Very broad, buzzword-heavy scope covering unrelated fields.
        - Suspiciously short submission-to-acceptance times.
        - Poorly written / template-like website.
        - Multiple similar conferences by same organizer with rotating locations.

        CLASSIFY into:
        - "likely reputable"
        - "uncertain"
        - "likely predatory or low-quality"

        Assign a **reputation score (0–100)**:
        - 80–100: clearly reputable
        - 60–79: somewhat reputable / mid-tier
        - 40–59: unclear / mixed signals
        - 0–39: likely predatory / low-quality

        OUTPUT:
        - Short summary line
        - Then markdown bullets, for example:

          - classification: ...
          - reputation_score: ...
          - positive_signals:
            - ...
          - negative_signals:
            - ...
          - evidence_links:
            - ...

        If evidence is weak, mark the result as **uncertain**.

        End with:
        "⚠️ This assessment is advisory only. Always cross-check with official
        indexing services and your advisor or colleagues."
        """,
    ),
    tools=[google_search],
)

conference_reputation_runner = InMemoryRunner(agent=conference_reputation_agent)

print("✅ Conference Reputation Agent & runner created.")


✅ Conference Reputation Agent & runner created.


### 4.4 PaperReviewAgent – venue-aware review

In [12]:
paper_review_agent = LlmAgent(
    name="paper_review_agent",
    model=gemini_model,
    instruction=textwrap.dedent(
        """
        You are a **venue-aware paper review assistant**.

        INPUT (via natural language):
        - Target conference name (e.g. "MICCAI 2026").
        - Paper description: title + abstract + methods + key results
          (typically produced by the PaperProfilerAgent).

        YOUR TASKS:

        1. Use `google_search` to understand the target conference:
           - scope & topics
           - typical paper style & level
           - any explicit review criteria, when visible
           - optionally, quick scan of past accepted paper titles/abstracts

        2. Analyze the paper description:
           - domain
           - main method(s)
           - datasets & experiments (if present)
           - key claims / improvements
           - obvious gaps (missing baselines, ablations, unclear methods)

        3. Produce a structured review with sections:

           1. **Short Summary** (3–5 sentences)
           2. **Strengths** (3–7 bullets, venue-aware)
           3. **Weaknesses / Concerns** (3–7 bullets, venue-aware)
           4. **Fit to the Conference** (1 paragraph)
           5. **Novelty & Impact (rough assessment)** (1 paragraph)
           6. **Concrete Suggestions Before Submission** (5–10 bullet points)
           7. **Overall Verdict (informal)** (1–2 sentences, e.g. "borderline but promising for MICCAI")

        RULES:
        - Be explicit when your assessment is limited by missing details.
        - Never pretend you saw the full paper if you only saw a summary.
        - Always ground venue information in `google_search` results, not pure memory.
        """,
    ),
    tools=[google_search],
)

paper_review_runner = InMemoryRunner(agent=paper_review_agent)

print("✅ Paper Review Agent & runner created.")

✅ Paper Review Agent & runner created.


## 5. ConfAIrCoordinator – top-level orchestrator

This agent uses **AgentTool** to call the specialist agents just like tools.

In [13]:
confair_coordinator = LlmAgent(
    name="confair_coordinator",
    model=gemini_model,
    instruction=textwrap.dedent(
        """
        You are **ConfAIrCoordinator**, the top-level orchestrator agent.

        You have access to the following tools/agents:

        - `pdf_to_text(path, max_pages)`         → extract text from a PDF.
        - `paper_profiler_agent`                 → summarize a paper & infer field.
        - `conference_finder_websearch`          → find upcoming conferences for a field.
        - `conference_reputation_agent`          → assess conference reputation/risk.
        - `paper_review_agent`                   → write a venue-aware review.

        HIGH-LEVEL BEHAVIOR:

        1. Understand the user's request.
           - They might:
             * provide a PDF path and ask for venues + review.
             * provide only text (title/abstract) and ask for venues.
             * provide a specific conference name and ask for a review.

        2. If the user gives a PDF path (e.g. "/kaggle/input/.../paper.pdf"):
           - Call `pdf_to_text` to extract text.
           - Then call `paper_profiler_agent` with a compact prompt including:
             * title_guess
             * abstract_guess
             * (optionally) some of the full_text and metadata.

        3. If the user instead gives a text description of the paper:
           - Skip `pdf_to_text` and directly call `paper_profiler_agent`
             with the provided description.

        4. Once you have a structured paper summary from the profiler:
           - Extract the `field` to pass to `conference_finder_websearch`.
           - Ask the Conference Finder for ~5–8 upcoming conferences.
           - For the top 3 conferences, call `conference_reputation_agent`
             to get reputation & risk signals.

        5. Select the BEST venue based on:
           - field match
           - reputation score (prefer reputable / top-tier)
           - the user's preference (prestige vs earliest deadline)
           - days_left if mentioned by the finder

        6. Call `paper_review_agent` for the selected venue:
           - Include:
             * target conference name
             * the structured summary from the profiler (title, abstract, methods, contributions).

        7. Compose a final answer for the user with:
           - A short explanation of the pipeline you followed.
           - A table of **recommended conferences** (from the finder).
           - A short bullet list summarizing the reputation of the top venues.
           - The **venue-aware review** for the top selected venue.
           - A reminder to double-check deadlines and details on official sites.

        STYLE:
        - Be concise but clear.
        - Use headings and bullet points for readability.
        - Explicitly mention that this is an advisory tool, not an official ranking.

        IMPORTANT:
        - Use the available tools; do not reinvent their work inside this agent.
        - You may call each tool multiple times as needed.
        """,
    ),
    tools=[
        pdf_to_text,
        AgentTool(agent=paper_profiler_agent),
        AgentTool(agent=conference_finder_agent),
        AgentTool(agent=conference_reputation_agent),
        AgentTool(agent=paper_review_agent),
    ],
)

confair_runner = InMemoryRunner(agent=confair_coordinator)

print("✅ ConfAIr Coordinator Agent & runner created.")

✅ ConfAIr Coordinator Agent & runner created.


## 6. Example: End-to-end run with a PDF

- Set `pdf_path` to point to your uploaded paper in /kaggle/input/...
- The coordinator will:
  1) parse the PDF,
  2) profile the paper,
  3) find conferences,
  4) check reputations,
  5) select the best,
  6) generate a venue-aware review.

In [14]:
import asyncio

# change this to the actual path of PDF
pdf_path = "/kaggle/input/your-dataset-folder/your_paper.pdf"

user_prompt = f"""
I want help deciding where to submit my paper and getting a venue-aware review.

Here is my paper PDF path:
{pdf_path}

Preferences:
- I care more about **prestige** than earliest deadline.
- Please:
  1) Analyze the paper,
  2) Find ~5–8 suitable conferences with upcoming deadlines,
  3) Check their reputation and filter out suspicious venues,
  4) Pick the best venue for me,
  5) Write a detailed review tailored to that venue.

Return everything in a structured, readable format with headings and tables.
"""

# Debug run: shows tool calls + final answer
_ = await confair_runner.run_debug(user_prompt)


 ### Created new session: debug_session_id

User > 
I want help deciding where to submit my paper and getting a venue-aware review.

Here is my paper PDF path:
/kaggle/input/your-dataset-folder/your_paper.pdf

Preferences:
- I care more about **prestige** than earliest deadline.
- Please:
  1) Analyze the paper,
  2) Find ~5–8 suitable conferences with upcoming deadlines,
  3) Check their reputation and filter out suspicious venues,
  4) Pick the best venue for me,
  5) Write a detailed review tailored to that venue.

Return everything in a structured, readable format with headings and tables.



confair_coordinator > The paper PDF path you provided does not seem to exist. Please double-check the path and try again. If the PDF is not accessible, I will not be able to analyze your paper or recommend suitable conferences.


## 7. Example: Run only on a text description (no PDF)

In [15]:

text_only_prompt = """
I don't have a PDF yet, but here is my paper description:

Title: Self-Supervised Transformer-Based Segmentation for Multi-Modal Breast Cancer MRI

Abstract:
We propose a self-supervised learning framework for 3D multi-modal breast MRI segmentation.
Our method uses a Vision Transformer backbone pre-trained with a masked volume modeling task
on 2,000 unlabeled breast MRI scans from three institutions. We then fine-tune on a labeled
dataset of 300 patients for tumor and fibroglandular tissue segmentation.

Compared to a strong nnUNet baseline, our method improves Dice score by 3.2 points for tumor
segmentation and 2.1 points for fibroglandular segmentation on an external test set from a
fourth institution. We conduct ablation studies to analyze the contribution of the
self-supervised pre-training and the multi-modal fusion strategy.

Preferences:
- Please recommend conferences with a focus on **prestige** rather than earliest deadline.
- Then generate a detailed review tailored to the top recommended venue.
"""

_ = await confair_runner.run_debug(text_only_prompt)


 ### Continue session: debug_session_id

User > 
I don't have a PDF yet, but here is my paper description:

Title: Self-Supervised Transformer-Based Segmentation for Multi-Modal Breast Cancer MRI

Abstract:
We propose a self-supervised learning framework for 3D multi-modal breast MRI segmentation.
Our method uses a Vision Transformer backbone pre-trained with a masked volume modeling task
on 2,000 unlabeled breast MRI scans from three institutions. We then fine-tune on a labeled
dataset of 300 patients for tumor and fibroglandular tissue segmentation.

Compared to a strong nnUNet baseline, our method improves Dice score by 3.2 points for tumor
segmentation and 2.1 points for fibroglandular segmentation on an external test set from a
fourth institution. We conduct ablation studies to analyze the contribution of the
self-supervised pre-training and the multi-modal fusion strategy.

Preferences:
- Please recommend conferences with a focus on **prestige** rather than earliest deadline.
- 

confair_coordinator > Here's a breakdown of the process and recommendations for your paper:

## Conference Recommendations & Review

### 1. Paper Analysis

I've analyzed your paper description, and here's a summary:

*   **Title:** Self-Supervised Transformer-Based Segmentation for Multi-Modal Breast Cancer MRI
*   **Abstract:** Proposes a self-supervised learning framework using a Vision Transformer (pre-trained with masked volume modeling) for 3D multi-modal breast MRI segmentation. It claims improved performance over a strong baseline for tumor and fibroglandular tissue segmentation, with ablation studies supporting the benefits of self-supervised pre-training and multi-modal fusion.
*   **Field:** Medical Image Analysis / Radiology
*   **Methods:** Self-Supervised Learning, Vision Transformer, Masked Volume Modeling, Multi-modal Fusion
*   **Contributions:** A novel self-supervised framework for breast cancer MRI segmentation, utilization of a pre-trained Vision Transformer, demons

## Demo: Text-Only Paper Description

In this example, I provide only the title and abstract of my paper (no PDF).
The ConfAIrCoordinator:

1. Calls the PaperProfilerAgent to infer the field and contributions.
2. Calls the ConferenceFinderAgent to find relevant, prestigious medical imaging conferences.
3. Calls the ConferenceReputationAgent on the top venues.
4. Selects MIUA as the best fit based on field and reputation.
5. Calls the PaperReviewAgent to generate a venue-aware review tailored to MIUA.

The table shows recommended conferences with deadlines and reputation,
and the final section is a venue-specific write-up for MIUA.